In [1]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Load endometriosis sample (with endometriosis)
endo = sc.read_10x_mtx(
    '/Users/jacob1/Desktop/biohackathon/OneDrive_1_13-05-2026/(Supplementary_Dataset)_Endometrium_single_cell_data/work/UA_Endo12604667_FX1130/',
    var_names='gene_symbols',
    cache=True
)
endo.obs['condition'] = 'Endometriosis'
endo.obs['sample'] = 'Endo12604667'

# Load normal endometrium sample
normal = sc.read_10x_mtx(
    '/Users/jacob1/Desktop/biohackathon/OneDrive_1_13-05-2026/(Supplementary_Dataset)_Endometrium_single_cell_data/work 2/UA_Endo12269811_FX0022/',
    var_names='gene_symbols',
    cache=True
)
normal.obs['condition'] = 'Normal'
normal.obs['sample'] = 'Endo12269811'

print("Endo cells:", endo.n_obs, "| Genes:", endo.n_vars)
print("Normal cells:", normal.n_obs, "| Genes:", normal.n_vars)

FileNotFoundError: Did not find file /Users/jacob1/Desktop/biohackathon/OneDrive_1_13-05-2026/(Supplementary_Dataset)_Endometrium_single_cell_data/work/UA_Endo12604667_FX1130/matrix.mtx.gz.

In [2]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Load endometriosis sample
endo = sc.read_10x_mtx(
    '/Users/jacob1/Desktop/biohackathon/OneDrive_1_13-05-2026/(Supplementary_Dataset)_Endometrium_single_cell_data/work/UA_Endo12604667_FX1130/',
    var_names='gene_symbols',
    cache=True
)
endo.obs['condition'] = 'Endometriosis'
endo.obs['sample'] = 'Endo12604667'

# Load normal endometrium sample
normal_endo = sc.read_10x_mtx(
    '/Users/jacob1/Desktop/biohackathon/OneDrive_1_13-05-2026/(Supplementary_Dataset)_Endometrium_single_cell_data/work/UA_Endo12269811_FX0022/',
    var_names='gene_symbols',
    cache=True
)
normal_endo.obs['condition'] = 'Normal'
normal_endo.obs['sample'] = 'Endo12269811'

print("Endo cells:", endo.n_obs, "| Genes:", endo.n_vars)
print("Normal cells:", normal_endo.n_obs, "| Genes:", normal_endo.n_vars)

Endo cells: 4844 | Genes: 36601
Normal cells: 5734 | Genes: 36601


In [4]:
# Merge both samples
adata_endo = ad.concat([endo, normal_endo], label='condition', keys=['Endometriosis', 'Normal'])
adata_endo.obs_names_make_unique()
print("Combined:", adata_endo.n_obs, "cells,", adata_endo.n_vars, "genes")

# QC filtering
sc.pp.filter_cells(adata_endo, min_genes=200)
sc.pp.filter_genes(adata_endo, min_cells=3)
print("After QC:", adata_endo.n_obs, "cells,", adata_endo.n_vars, "genes")

# Normalize and log transform
sc.pp.normalize_total(adata_endo, target_sum=1e4)
sc.pp.log1p(adata_endo)

# Check endometriosis marker genes from literature + your groupmate's heatmap
target_genes_endo = ['ESR1', 'SOX5', 'MECOM', 'TCF12', 'ZBTB20', 'STAR', 'IGFBP4', 'HTRA1', 'KRT18', 'CYP11A1']
found = [g for g in target_genes_endo if g in adata_endo.var_names]
missing = [g for g in target_genes_endo if g not in adata_endo.var_names]
print("\nFound genes:", found)
print("Missing genes:", missing)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Combined: 10578 cells, 36601 genes
After QC: 10578 cells, 30337 genes

Found genes: ['ESR1', 'SOX5', 'MECOM', 'TCF12', 'ZBTB20', 'STAR', 'IGFBP4', 'HTRA1', 'KRT18', 'CYP11A1']
Missing genes: []


In [5]:
# Statistical test for each gene
results_endo = []
for gene in target_genes_endo:
    endo_expr = adata_endo[adata_endo.obs['condition'] == 'Endometriosis', gene].X.toarray().flatten()
    normal_expr = adata_endo[adata_endo.obs['condition'] == 'Normal', gene].X.toarray().flatten()
    
    stat, pval = stats.mannwhitneyu(endo_expr, normal_expr, alternative='two-sided')
    
    endo_mean = endo_expr.mean()
    normal_mean = normal_expr.mean()
    fold_change = endo_mean - normal_mean
    
    results_endo.append({
        'Gene': gene,
        'Endo_mean': round(endo_mean, 4),
        'Normal_mean': round(normal_mean, 4),
        'Log_FC': round(fold_change, 4),
        'Direction': '↑ Endo' if fold_change > 0 else '↓ Endo',
        'p_value': round(pval, 6),
        'Significant': '✅' if pval < 0.05 else '❌'
    })

df_endo = pd.DataFrame(results_endo)
print(df_endo.to_string(index=False))

   Gene  Endo_mean  Normal_mean  Log_FC Direction  p_value Significant
   ESR1     3.8871       3.5986  0.2884    ↑ Endo 0.000000           ✅
   SOX5     2.9866       2.8278  0.1589    ↑ Endo 0.000000           ✅
  MECOM     2.6594       2.7500 -0.0906    ↓ Endo 0.000263           ✅
  TCF12     2.3521       2.5787 -0.2265    ↓ Endo 0.000000           ✅
 ZBTB20     2.8746       3.0135 -0.1389    ↓ Endo 0.000000           ✅
   STAR     0.0179       0.0196 -0.0017    ↓ Endo 0.871499           ❌
 IGFBP4     0.7315       0.5888  0.1427    ↑ Endo 0.000000           ✅
  HTRA1     0.3464       0.5238 -0.1775    ↓ Endo 0.000000           ✅
  KRT18     0.1834       0.1533  0.0301    ↑ Endo 0.000000           ✅
CYP11A1     0.0106       0.0069  0.0037    ↑ Endo 0.026858           ✅
